# Week 5 — Eval harness v2

The week 4 harness worked but the smoke test showed up three things, and reading round the
standard RAG evaluation metrics showed up a fourth:

1. **refusals were being scored for faithfulness.** "I don't have reliable information to answer
   that" got marked unsupported, which is nonsense - a refusal has nothing in the passages to be
   faithful to. That's what dragged faithfulness from 0.79 (medical questions) down to 0.68.
2. **the baseline couldn't be scored on faithfulness at all**, because it never sees passages. So
   I could never say "RAG hallucinates less than the baseline", which is the whole argument.
3. **no retrieval metrics.** Week 3 checked retrieval by eye. The standard set (RAGAS, and the
   IR metrics everyone reports) has context precision on the retrieval side, and Baur et al
   reported it. I had the generation half and none of the retrieval half.
4. **the judge might be unfair to simplified text**, which would wreck the week 6 experiment
   before it starts. See the calibration section.

So v2 adds: hit rate@k and MRR (free, because every test question carries its source topic),
context precision, refusal detection, baseline faithfulness against would-be-retrieved passages,
and a calibration block that checks the judge on simplified answers as well as normal ones.

Week 4's notebook stays as it is - it's the record of what I ran that week. This is the harness
everything from here on uses.

In [1]:
import json
import re
import time
from collections import Counter
from datetime import datetime, timezone

import chromadb
import ollama
import pandas as pd
import textstat
from sentence_transformers import SentenceTransformer

/Users/charlesbruce/Desktop/UWE/venv-arm64/lib/python3.13/site-packages/textstat/textstat.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/charlesbruce/Desktop/UWE/venv-arm64/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
GENERATOR_MODEL = 'mistral:7b-instruct'
JUDGE_MODEL = 'llama3.1:8b'          # different model on purpose - see week 4 justification 7
TEMPERATURE = 0
SEED = 42                            # belt and braces on top of temp 0

DB_DIR = 'chroma_db'
COLLECTION = 'medlineplus_english'
EMBED_MODEL = 'BAAI/bge-small-en-v1.5'
TOP_K = 5
MAX_PER_TOPIC = 2

RUN_LOG = 'wk5_runs.jsonl'

REFUSAL_MARKER = "i don't have reliable information"   # lowercase, matched loosely

In [3]:
def generate(prompt, model=GENERATOR_MODEL):
    resp = ollama.chat(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': TEMPERATURE, 'seed': SEED},
    )
    return resp['message']['content'].strip()


embedder = SentenceTransformer(EMBED_MODEL)
coll = chromadb.PersistentClient(path=DB_DIR).get_collection(COLLECTION)
print('index loaded:', coll.count(), 'chunks')


def retrieve(query, k=TOP_K, max_per_topic=MAX_PER_TOPIC):
    q_emb = embedder.encode('Represent this sentence for searching relevant passages: ' + query)
    res = coll.query(query_embeddings=[q_emb.tolist()], n_results=k * 10)

    hits, spill, per_topic = [], [], Counter()
    for doc, meta, dist in zip(res['documents'][0], res['metadatas'][0], res['distances'][0]):
        hit = {'text': doc, 'title': meta['title'], 'url': meta['url'],
               'group': meta['group'], 'similarity': round(1 - dist, 3)}
        if per_topic[meta['topic_id']] >= max_per_topic:
            spill.append(hit)
            continue
        per_topic[meta['topic_id']] += 1
        hits.append(hit)
        if len(hits) == k:
            break
    if len(hits) < k:
        hits.extend(spill[:k - len(hits)])
    return hits

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6693.13it/s]


index loaded: 10337 chunks


## Retrieval metrics

Every test question carries the topic it came from, so I know which topic *should* be retrieved.
That gives me ground-truth retrieval metrics for free:

- **hit rate@k** - did the right topic show up in the top k at all
- **MRR** - how high up was it (1.0 if first, 0.5 if second, and so on)

Context precision is different - it asks how many of the retrieved chunks are actually useful for
this question, which needs judgement rather than a lookup, so the judge model scores each chunk.
That's the reference-free version of the RAGAS metric.

Worth remembering when I read these: the questions mostly contain the topic name, so hit rate will
look flattering. It's still useful for comparing k values against each other.

In [4]:
def retrieval_metrics(hits, gold_title):
    titles = [h['title'] for h in hits]
    if gold_title in titles:
        rank = titles.index(gold_title) + 1
        return {'hit': 1, 'rank': rank, 'rr': round(1 / rank, 3)}
    return {'hit': 0, 'rank': None, 'rr': 0.0}


CONTEXT_PROMPT = """A patient asked: "{question}"

Here is one passage that was retrieved to help answer it:
\"\"\"{chunk}\"\"\"

Does this passage contain information that helps answer the patient's question? Answer YES if it is useful, NO if it is off-topic or unhelpful.

Reply with exactly one word: YES or NO."""


def context_precision(question, hits):
    """share of retrieved chunks the judge considers useful for this question."""
    if not hits:
        return {'context_precision': None, 'useful_chunks': 0}
    verdicts = []
    for h in hits:
        out = generate(CONTEXT_PROMPT.format(question=question, chunk=h['text']), model=JUDGE_MODEL)
        verdicts.append(out.strip().upper().startswith('YES'))
    return {'context_precision': round(sum(verdicts) / len(verdicts), 2),
            'useful_chunks': sum(verdicts)}

## Generation

Same grounding prompt as week 4, but with a reading-level slot so week 6 can vary it without
changing anything else. `None` means no instruction at all, which is the uncontrolled condition.

In [5]:
RAG_PROMPT = """You are a patient education assistant. Answer the patient's question using ONLY the information in the passages below.

Rules:
- Use only facts stated in the passages. Do not add anything from your own knowledge.
- After each fact, cite the passage it came from by its topic title in square brackets, like [Heart Failure].
{reading_level}- If the passages do not contain the information needed to answer, say exactly: \"I don't have reliable information to answer that. Please ask your health care provider.\"
- Do not give a diagnosis. Do not recommend changing any medicine or dose.

Passages:
{passages}

Patient's question: {question}

Answer:"""

BASELINE_PROMPT = """You are a patient education assistant. Answer the patient's question.

Rules:
{reading_level}- Do not give a diagnosis. Do not recommend changing any medicine or dose.

Patient's question: {question}

Answer:"""

READING_LEVELS = {
    'grade6': '- Write so a 6th grade reader can understand it. Use short sentences and simple words.\n',
    'grade9': '- Write so a 9th grade reader can understand it.\n',
    'none':   '- Write in plain, everyday language a patient can understand.\n',
}


def format_passages(hits):
    return '\n\n---\n\n'.join(f"Topic: {h['title']}\n{h['text']}" for h in hits)


def rag_answer(question, k=TOP_K, level='none'):
    hits = retrieve(question, k=k)
    prompt = RAG_PROMPT.format(passages=format_passages(hits), question=question,
                               reading_level=READING_LEVELS[level])
    return {'question': question, 'answer': generate(prompt), 'hits': hits, 'level': level}


def baseline_answer(question, k=TOP_K, level='none'):
    """no passages in the prompt - but I still retrieve them, so faithfulness can be scored
    against what the system WOULD have had. that's the fix for week 4's blind spot."""
    prompt = BASELINE_PROMPT.format(question=question, reading_level=READING_LEVELS[level])
    return {'question': question, 'answer': generate(prompt),
            'hits': retrieve(question, k=k), 'level': level}

## Generation metrics

Readability and citation precision are pure calculation. Faithfulness and relevancy use the judge.

The refusal fix: if an answer is a refusal, faithfulness is recorded as `None` rather than 0, and
the refusal is counted separately. Refusing an unanswerable question is correct behaviour, not a
hallucination, and averaging it in as a zero punishes the system for doing the right thing.

In [6]:
CITE_RE = re.compile(r'\[([^\]]+)\]')


def strip_citations(text):
    return re.sub(r'\s{2,}', ' ', CITE_RE.sub('', text)).strip()


def is_refusal(answer):
    return REFUSAL_MARKER in answer.lower()


def readability(answer):
    clean = strip_citations(answer)
    if len(clean.split()) < 10:      # textstat is unstable on very short text
        return {'fkgl': None, 'fre': None, 'words': len(clean.split())}
    return {'fkgl': round(textstat.flesch_kincaid_grade(clean), 1),
            'fre': round(textstat.flesch_reading_ease(clean), 1),
            'words': len(clean.split())}


def citation_scores(answer, hits):
    cited = [c.strip() for c in CITE_RE.findall(answer)]
    if not cited:
        return {'cited_any': False, 'citation_precision': None, 'n_citations': 0}
    retrieved = {h['title'].lower() for h in hits}
    valid = sum(1 for c in cited if c.lower() in retrieved)
    return {'cited_any': True, 'citation_precision': round(valid / len(cited), 2),
            'n_citations': len(cited)}


def split_sentences(text):
    clean = strip_citations(text)
    return [p.strip() for p in re.split(r'(?<=[.!?])\s+', clean) if len(p.strip()) > 20]

In [7]:
# the judge prompt now says a reworded sentence still counts. without that line the judge
# penalises simplified answers just for using different words, which would fake a
# readability/faithfulness trade-off in week 6. the calibration section below tests whether
# this actually holds.
FAITHFUL_PROMPT = """Here are passages from a medical reference, and one sentence from an answer that was written using them.

Passages:
{passages}

Sentence: \"{sentence}\"

Is the sentence supported by the passages? A sentence counts as supported if its factual claims appear in the passages, even if it is worded differently or uses simpler words. General advice to consult a doctor counts as supported.

Reply with exactly one word: YES or NO."""


def judge_faithfulness(record):
    if is_refusal(record['answer']):
        return {'faithfulness': None, 'n_sentences': 0, 'unsupported': [], 'refused': True}

    sentences = split_sentences(record['answer'])
    if not sentences:
        return {'faithfulness': None, 'n_sentences': 0, 'unsupported': [], 'refused': False}

    passages = format_passages(record['hits'])
    verdicts, unsupported = [], []
    for s in sentences:
        out = generate(FAITHFUL_PROMPT.format(passages=passages, sentence=s), model=JUDGE_MODEL)
        ok = out.strip().upper().startswith('YES')
        verdicts.append(ok)
        if not ok:
            unsupported.append(s)
    return {'faithfulness': round(sum(verdicts) / len(verdicts), 2),
            'n_sentences': len(sentences), 'unsupported': unsupported, 'refused': False}


RELEVANCY_PROMPT = """A patient asked: \"{question}\"

They received this answer:
\"{answer}\"

On a scale of 1 to 5, how well does the answer address the patient's actual question? 1 = not at all, 3 = partially, 5 = fully addresses it.

Reply with only the number."""


def judge_relevancy(record):
    out = generate(RELEVANCY_PROMPT.format(question=record['question'],
                                           answer=strip_citations(record['answer'])),
                   model=JUDGE_MODEL)
    m = re.search(r'[1-5]', out)
    return {'relevancy': int(m.group()) if m else None}

In [8]:
def score(record, arm, gold_title=None, with_context_precision=True):
    """score one answer. both arms now get faithfulness - the baseline is judged against the
    passages it WOULD have been given, which is what makes the two arms comparable."""
    s = {'arm': arm, 'level': record.get('level', 'none')}
    s.update(readability(record['answer']))
    s.update(judge_relevancy(record))
    s.update(judge_faithfulness(record))

    if arm == 'rag':
        s.update(citation_scores(record['answer'], record['hits']))

    if gold_title is not None:
        s.update(retrieval_metrics(record['hits'], gold_title))
    if with_context_precision:
        s.update(context_precision(record['question'], record['hits']))
    return s


def log_run(record, scores, extra=None, log_path=RUN_LOG):
    entry = {
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'question': record['question'],
        'answer': record['answer'],
        'retrieved': [{'title': h['title'], 'url': h['url'], 'similarity': h['similarity']}
                      for h in record['hits']],
        'scores': scores,
        'config': {'generator': GENERATOR_MODEL, 'judge': JUDGE_MODEL,
                   'temperature': TEMPERATURE, 'seed': SEED, 'k': TOP_K,
                   'max_per_topic': MAX_PER_TOPIC, 'embed_model': EMBED_MODEL,
                   'level': record.get('level', 'none')},
        **(extra or {}),
    }
    with open(log_path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    return entry

## Timing check

Before committing to a big run I want real numbers, not the 13s from week 4 which included model
loading. Judge calls dominate - one answer costs 1 generation plus roughly (sentences + k) judge
calls - so this tells me whether the week 6 experiment is an overnight job or a rethink.

In [9]:
dev = json.load(open('test_queries_dev.json', encoding='utf-8'))
heldout = json.load(open('test_queries_heldout.json', encoding='utf-8'))
print(f'dev {len(dev)} | held-out {len(heldout)}')

generate('warm up the model')   # first call loads weights, don't time that one

t0 = time.time()
r = rag_answer(dev[0]['question'])
t_gen = time.time() - t0

t0 = time.time()
s = score(r, 'rag', gold_title=dev[0]['topic_title'])
t_score = time.time() - t0

print(f'\ngeneration {t_gen:.1f}s | scoring {t_score:.1f}s | total {t_gen + t_score:.1f}s per answer')
n_runs = len(dev) * 3 * 2      # 3 reading levels x 2 arms
print(f'week 6 full dev run is ~{n_runs} answers -> {n_runs * (t_gen + t_score) / 3600:.1f} hours')
print(f'\nQ: {r["question"]}')
print(f'retrieved: {[h["title"] for h in r["hits"]]}')
print(f'gold topic: {dev[0]["topic_title"]}')
print(f'\nscores: {json.dumps({k: v for k, v in s.items() if k != "unsupported"}, indent=1)}')
print(f'\nanswer:\n{r["answer"][:600]}')

dev 115 | held-out 85

generation 21.5s | scoring 34.0s | total 55.4s per answer
week 6 full dev run is ~690 answers -> 10.6 hours

Q: What are the treatments for Transient Ischemic Attack?
retrieved: ['Ischemic Stroke', 'Ischemic Stroke', 'Transient Ischemic Attack', 'Stroke', 'Transient Ischemic Attack']
gold topic: Transient Ischemic Attack

scores: {
 "arm": "rag",
 "level": "none",
 "fkgl": 10.9,
 "fre": 51.6,
 "words": 100,
 "relevancy": 4,
 "faithfulness": 1.0,
 "n_sentences": 5,
 "refused": false,
 "cited_any": true,
 "citation_precision": 1.0,
 "n_citations": 2,
 "hit": 1,
 "rank": 3,
 "rr": 0.333,
 "context_precision": 0.6,
 "useful_chunks": 3
}

answer:
The treatment for a Transient Ischemic Attack (TIA), often called a mini-stroke, usually involves medicines. However, unlike ischemic stroke treatments, there isn't a medicine like tPA (tissue plasminogen activator) to dissolve the blood clot in a TIA because it lasts only a few minutes [Transient Ischemic Attack]. Instead, i

## Judge calibration — normal AND simplified answers

This is the bit that protects week 6. The experiment asks whether simplifying an answer costs
faithfulness. But a 6th-grade rewrite reuses fewer of the source's words, so a strict judge might
mark a perfectly good paraphrase as unsupported. If that happens I'd "find" a trade-off that is
really just the metric reacting to rewording - and the whole result would be worthless.

So I calibrate on both conditions. Fill `my_verdict` by hand (1 supported / 0 not) without looking
at the judge column, then run the agreement cell. What matters is not just overall agreement but
whether agreement is **similar for normal and simplified answers**. If it drops for simplified
text, the metric is biased and the judge prompt needs work before running the experiment.

In [10]:
CALIB_N = 8      # questions per condition - 8 gives roughly 40-60 sentences to rate by hand

rows = []
for q in dev[:CALIB_N]:
    for level in ['none', 'grade6']:
        rec = rag_answer(q['question'], level=level)
        if is_refusal(rec['answer']):
            continue
        passages = format_passages(rec['hits'])
        for s in split_sentences(rec['answer']):
            out = generate(FAITHFUL_PROMPT.format(passages=passages, sentence=s), model=JUDGE_MODEL)
            rows.append({
                'level': level,
                'question': q['question'],
                'sentence': s,
                'judge_verdict': 1 if out.strip().upper().startswith('YES') else 0,
                'my_verdict': '',
                'passages': passages,
            })

pd.DataFrame(rows).to_csv('judge_calibration_v2.csv', index=False)
print(f'wrote {len(rows)} sentences to judge_calibration_v2.csv')
print(Counter(r['level'] for r in rows))
print('\nfill my_verdict (1/0) WITHOUT looking at judge_verdict, then run the next cell')

wrote 48 sentences to judge_calibration_v2.csv
Counter({'none': 25, 'grade6': 23})

fill my_verdict (1/0) WITHOUT looking at judge_verdict, then run the next cell


In [11]:
cal = pd.read_csv('judge_calibration_v2.csv')
done = cal[cal['my_verdict'].notna() & (cal['my_verdict'].astype(str).str.strip() != '')].copy()

if len(done) == 0:
    print('my_verdict not filled yet')
else:
    done['my_verdict'] = done['my_verdict'].astype(int)
    done['agree'] = done['judge_verdict'] == done['my_verdict']

    print(f'overall agreement: {done["agree"].mean():.0%} on {len(done)} sentences\n')
    print('by reading level - these two numbers should be close:')
    for level, grp in done.groupby('level'):
        print(f'  {level:8s}  {grp["agree"].mean():.0%}  ({len(grp)} sentences)')

    gaps = done.groupby('level')['agree'].mean()
    if len(gaps) == 2 and abs(gaps.iloc[0] - gaps.iloc[1]) > 0.15:
        print('\n>>> WARNING: agreement differs by more than 15 points between conditions.')
        print('    The judge is treating simplified text differently. Fix the judge prompt')
        print('    before running the week 6 experiment, or the result will be an artifact.')
    else:
        print('\nagreement is similar across conditions - metric looks fair to simplified text')

    print('\ndisagreements:')
    for _, r in done[~done['agree']].iterrows():
        print(f'  [{r["level"]}] judge={r["judge_verdict"]} me={r["my_verdict"]}: {r["sentence"][:85]}')

my_verdict not filled yet


## Pilot — both arms on a slice of dev

Small run to confirm every metric populates and the RAG/baseline comparison works end to end,
including baseline faithfulness. Not a result, just a check before the full runs.

In [12]:
PILOT_N = 10
results = []

for i, q in enumerate(dev[:PILOT_N], 1):
    print(f'[{i}/{PILOT_N}] {q["question"][:60]}')
    for arm, fn in [('rag', rag_answer), ('baseline', baseline_answer)]:
        rec = fn(q['question'])
        s = score(rec, arm, gold_title=q['topic_title'])
        log_run(rec, s, extra={'topic_title': q['topic_title'], 'group': q['group'],
                               'qtype': q['qtype'], 'split': 'dev'})
        results.append({'question': q['question'], 'group': q['group'],
                        **{k: v for k, v in s.items() if k != 'unsupported'}})

df = pd.DataFrame(results)
df.to_csv('wk5_pilot_results.csv', index=False)
print(f'\ndone - {len(df)} rows')

[1/10] What are the treatments for Transient Ischemic Attack?
[2/10] What is (are) Scleroderma?
[3/10] What is (are) Skin Conditions?
[4/10] What are the treatments for Carpal Tunnel Syndrome?
[5/10] How many people are affected by Hemorrhoids?
[6/10] What are the treatments for Gaucher Disease?
[7/10] Do you have information about Rehabilitation
[8/10] What are the treatments for Myasthenia Gravis?
[9/10] What are the symptoms of Abdominal Adhesions?
[10/10] Do you have information about Health Screening

done - 20 rows


In [13]:
summary = df.groupby('arm').agg(
    faithfulness=('faithfulness', 'mean'),
    context_precision=('context_precision', 'mean'),
    hit_rate=('hit', 'mean'),
    mrr=('rr', 'mean'),
    relevancy=('relevancy', 'mean'),
    fkgl=('fkgl', 'mean'),
    words=('words', 'mean'),
    citation_precision=('citation_precision', 'mean'),
    refused=('refused', 'sum'),
).round(2)
print(summary)

print('\nthe comparison week 4 could not make:')
for arm in ['rag', 'baseline']:
    f = df[df['arm'] == arm]['faithfulness']
    print(f'  {arm:9s} faithfulness {f.mean():.2f}  (scored on {f.notna().sum()} of {len(f)} answers)')

          faithfulness  context_precision  hit_rate   mrr  relevancy   fkgl  \
arm                                                                           
baseline          0.84                0.6       1.0  0.93        4.4  10.77   
rag               0.97                0.6       1.0  0.93        3.8  11.13   

          words  citation_precision  refused  
arm                                           
baseline  192.5                 NaN        0  
rag        76.0                0.98        1  

the comparison week 4 could not make:
  rag       faithfulness 0.97  (scored on 9 of 10 answers)
  baseline  faithfulness 0.84  (scored on 10 of 10 answers)
